# 03. 대시보드 산출물 만들기전체 시즌 데이터로 **참조 테이블**과 **선수·팀 프로파일**을 만든다.모델 학습과 무관하므로 **GPU 불필요**(CPU 런타임으로 충분).만드는 것| 파일 | 쓰임 ||---|---|| `tables.pkl` | 투수×구종 실측 평균, 카운트별 결과 가치, 위치별 가치 → 추천 엔진 || `profiles.pkl` | 선수 명부·팀 소속, 타자/투수 존 히트맵, 카운트별 구사율 → 대시보드 || `batter_stats.pkl` | 타자 직전 시즌 성적 → 모델 맥락 피처 |1.4M 행을 로컬로 내려받는 것보다 여기서 만들어 몇 MB만 받는 편이 빠르다.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

In [ ]:
# src/ 코드 가져오기 (02_train.ipynb 와 동일)import pathlib, zipfileREPO = pathlib.Path('/content/repo')if not (REPO / 'src').exists():    from google.colab import files    uploaded = files.upload()          # colab_src.zip 선택    REPO.mkdir(parents=True, exist_ok=True)    with zipfile.ZipFile(next(iter(uploaded))) as z:        z.extractall(REPO)%cd /content/repo!pip install -q scikit-learn joblib

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, '/content/repo')import numpy as npimport pandas as pdimport joblibfrom src.data import (OUTCOME_TO_IDX, build_context_features, build_pitch_features,                      compute_batter_stats, define_outcome, load_raw)from src.features import build_reference_tablesfrom src.profiles import build_profilesDRIVE = Path('/content/drive/MyDrive/baseball_ai')RAW = DRIVE / 'data' / 'raw'OUT = DRIVE / 'artifacts'OUT.mkdir(parents=True, exist_ok=True)paths = sorted(RAW.glob('*.parquet'))print('원본 파일', len(paths), '개')

## 전처리

In [ ]:
df = load_raw(paths)df['outcome'] = define_outcome(df)df = df[df['outcome'].notna()].reset_index(drop=True)df['outcome_idx'] = df['outcome'].map(OUTCOME_TO_IDX).astype(np.int64)df = build_pitch_features(df)batter_stats = compute_batter_stats(df)df = build_context_features(df, batter_stats)print(f'{len(df):,}구')print('시즌별:', df.season.value_counts().sort_index().to_dict())print('결과 분포:')print((df.outcome.value_counts(normalize=True) * 100).round(2).to_string())

## 참조 테이블추천 엔진이 쓴다. 배포용 앱이므로 **두 시즌 전체**로 만든다(2025 매치업을 추천하려면 2025 레퍼토리가 필요하다).반사실 **평가**를 할 때는 테스트 시즌 정보가 섞이면 안 되므로,`scripts/build_tables.py --seasons 2024` 로 따로 만들어 쓴다.

In [ ]:
tables = build_reference_tables(df)tables.save(OUT / 'tables.pkl')joblib.dump(batter_stats, OUT / 'batter_stats.pkl')rep = tables.repertoireprint(f"(투수, 구종) 조합 {len(rep):,} / 투수 {rep.pitcher.nunique():,}명")print(f"투수당 평균 구종 수 {rep.groupby('pitcher').size().mean():.1f}")print(f"타자 성적 {len(batter_stats):,}행")

## 선수·팀 프로파일`min_pitches` 는 화면에 노출할 최소 표본이다. 풀시즌 기준 250구면대략 60타석 정도로, 지표가 어느 정도 안정되는 하한이다.

In [ ]:
name_map = {}p = Path('/content/repo/models/legacy/player_mapping.pkl')if p.exists():    name_map = joblib.load(p)    print(f'이름 매핑 {len(name_map):,}건')profiles = build_profiles(df, name_map, min_pitches=250)profiles.save(OUT / 'profiles.pkl')d = profiles.directoryprint(f'명부: 투수 {(d.role=="pitcher").sum():,}행 / 타자 {(d.role=="batter").sum():,}행')print(f'분석 가능: 타자 {len(profiles.batter_summary):,}명 / 투수 {len(profiles.pitcher_summary):,}명')print(f'이름 미확인 {d.name.str.startswith("ID ").sum():,}건')print(f'타자 표본 중앙값 {profiles.batter_summary.pitches.median():.0f}구')

### 선수 이름 보강 (선택)`player_mapping.pkl` 은 2024 기준이라 2025 신인이 빠져 있다.pybaseball 로 조회해 채운다. 실패해도 대시보드는 ID 로 표시하므로 건너뛰어도 된다.

In [ ]:
missing = d[d.name.str.startswith('ID ')].player_id.astype(int).unique().tolist()print(f'이름 미확인 {len(missing)}명')if missing:    !pip install -q pybaseball    from pybaseball import playerid_reverse_lookup    found = playerid_reverse_lookup(missing, key_type='mlbam')    extra = {int(r.key_mlbam): f"{r.name_first.title()} {r.name_last.title()}"             for r in found.itertuples() if pd.notna(r.name_last)}    print(f'{len(extra)}명 이름 확인')    name_map = {**{int(k): v for k, v in name_map.items()}, **extra}    joblib.dump(name_map, OUT / 'player_mapping.pkl')    profiles = build_profiles(df, name_map, min_pitches=250)    profiles.save(OUT / 'profiles.pkl')    d = profiles.directory    print(f'남은 미확인 {d.name.str.startswith("ID ").sum():,}건')

## 모델 평가 (학습 산출물이 있을 때만)`02_train.ipynb` 로 만든 `artifacts/main/` 이 있으면, 테스트 시즌에서혼동 행렬·클래스별 지표·캘리브레이션을 계산해 `evaluation.pkl` 로 저장한다.대시보드의 '모델 성능' 페이지와 보고서 그림이 이걸 쓴다.

In [ ]:
import jsonfrom torch.utils.data import DataLoaderfrom src.evaluate import evaluate_full, summary_textfrom src.model import ContextAwareTransformer, ModelConfigfrom src.prepare import preparefrom src.train import TrainConfig, temporal_val_split, to_tensorsimport torchMAIN = OUT / 'main'if not (MAIN / 'model_config.json').exists():    print('학습 산출물이 없습니다. 02_train.ipynb 를 먼저 실행하세요.')else:    cfg = ModelConfig(**json.loads((MAIN / 'model_config.json').read_text(encoding='utf-8')))    model = ContextAwareTransformer(cfg)    model.load_state_dict(torch.load(MAIN / 'model.pth', map_location='cpu'))    model = model.to('cuda' if torch.cuda.is_available() else 'cpu').eval()    device = 'cuda' if torch.cuda.is_available() else 'cpu'    # 학습과 완전히 같은 전처리·분할을 재현한다    train_seq, test_seq, _ = prepare(paths, train_seasons=[2024], test_seasons=[2025],                                     seq_len=cfg.seq_len)    _, va_idx = temporal_val_split(train_seq, TrainConfig().val_fraction)    test_loader = DataLoader(to_tensors(test_seq), batch_size=2048)    val_loader = DataLoader(to_tensors(train_seq, va_idx), batch_size=2048)    result = evaluate_full(model, test_loader, val_loader, device=device)    joblib.dump(result, MAIN / 'evaluation.pkl')    print(summary_text(result))    print(f"저장: {MAIN / 'evaluation.pkl'}")

## 내려받기

In [ ]:
import shutil, osfor f in ['tables.pkl', 'profiles.pkl', 'batter_stats.pkl', 'player_mapping.pkl',
          'main/evaluation.pkl', 'main/model.pth', 'main/encoders.pkl']:    p = OUT / f    if p.exists():        print(f'{f:22s} {p.stat().st_size/1e6:6.1f} MB')shutil.make_archive('/content/dashboard_artifacts', 'zip', OUT,                    base_dir=None)print(f"\nzip {os.path.getsize('/content/dashboard_artifacts.zip')/1e6:.1f} MB")from google.colab import filesfiles.download('/content/dashboard_artifacts.zip')

## 로컬에 넣기받은 zip 을 풀어 저장소의 `models/` 에 넣는다.```models/├── tables.pkl├── profiles.pkl├── batter_stats.pkl├── player_mapping.pkl└── main/            ← 02_train.ipynb 산출물 (model.pth, encoders.pkl, ...)```그다음 `streamlit run app.py` 를 다시 실행하면 전체 선수 데이터가 들어간다.(Streamlit 캐시가 남아 있으면 앱 우상단 메뉴 → **Clear cache** 후 새로고침)